**Where the data comes from.** `results/gauntlet/from_run_N` (the agents of GA `run_N`; a repeat gets a `_2` suffix) is written by `python AI/GA/gauntlet.py`: every agent plays the same seeded simulator games, `TimeSurvived` is virtual game time, `Died` says whether the agent topped out, `Seed` lets agents be compared game-by-game, and a `baseline` agent (your default weights) is the reference. Each run folder also has `summary.csv` and `config.json`. The cell below analyses the newest run; use `analyze_agents("from_run_3")` for another folder.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob

# --- project paths (work from any location) ---
from pathlib import Path


def find_project_root(start=None):
    """Walk up from the notebook's folder until the folder containing AI/ and Tetris/ is found."""
    here = Path(start or Path.cwd()).resolve()
    for d in [here, *here.parents]:
        if (d / "AI").is_dir() and (d / "Tetris").is_dir():
            return d
    raise FileNotFoundError("Open this notebook from inside the Tetris-Project folder.")


ROOT = find_project_root()
GA_DIR = ROOT / "results" / "GA"
GAUNTLET_DIR = ROOT / "results" / "gauntlet"


def latest_run(base_dir):
    """Most recently finished gauntlet folder (from_run_N, set_<name> or an old run_N)."""
    runs = [d for d in Path(base_dir).iterdir() if d.is_dir() and (d / "summary.csv").exists()]
    if not runs:
        raise FileNotFoundError(f"No gauntlet result folders in {base_dir}")
    return max(runs, key=lambda d: (d / "summary.csv").stat().st_mtime)


def analyze_agents(run_name, base_dir=GAUNTLET_DIR):
    # run_name: a folder name such as "from_run_4", or a number N for an old run_N folder
    run_name = run_name if isinstance(run_name, str) else f"run_{run_name}"
    run_folder = os.path.join(base_dir, run_name)
    # Accept ANY csv file that starts with "results_"
    file_pattern = os.path.join(run_folder, "results_*.csv")
    agent_files = glob.glob(file_pattern)
    
    agents = {}
    for file_path in agent_files:
        df = pd.read_csv(file_path)
        # Use AgentName if exists, otherwise fallback to filename
        if "AgentName" in df.columns:
            agent_name = df["AgentName"].iloc[0]
        else:
            agent_name = os.path.splitext(os.path.basename(file_path))[0]
        agents[agent_name] = df
    if not agents:
        raise FileNotFoundError(f"No agent result files found in {run_folder} (pattern {file_pattern}).")

    summary = []
    for name, df in agents.items():
        total_games = len(df)
        if "Died" in df.columns:
            # simulator runs: survived the whole budget = never topped out
            games_full_survival = (df["Died"] == 0).sum()
        else:
            # old real-time runs: full survival = reached the 600 s timeout
            games_full_survival = (df["TimeSurvived"] >= 600).sum()
        summary.append({
            "Agent": name,
            "MeanScore": df["Score"].mean(),
            "MedianScore": df["Score"].median(),
            "StdScore": df["Score"].std(),
            "MeanTetrises": df["NumTetrises"].mean(),
            "MaxTetrises": df["NumTetrises"].max(),
            "GamesFullSurvival": games_full_survival,
            "SurvivalRate": games_full_survival / total_games if total_games > 0 else 0,
            "GamesUnder20k": (df["Score"] < 20000).sum(),
            "LowestScore": df["Score"].min(),
            "HighestScore": df["Score"].max(),
        })

    summary_df = pd.DataFrame(summary)

    # Paired comparison against the reference agent: every agent played the same seeded games,
    # so score differences per seed are fair. Reference = "baseline" (added by the gauntlet) or
    # "m1" (the default weights, in older sets). Needs the Seed column (simulator runs).
    ref_name = next((n for n in ("baseline", "m1") if n in agents), None)
    if ref_name is not None and all("Seed" in d.columns for d in agents.values()):
        ref_scores = agents[ref_name].set_index("Seed")["Score"]
        diffs, better = {}, {}
        for name, df in agents.items():
            if name == ref_name:
                continue
            d = df.set_index("Seed")["Score"] - ref_scores
            diffs[name], better[name] = d.mean(), int((d > 0).sum())
        summary_df["VsRefMeanDiff"] = summary_df["Agent"].map(diffs).round(1)
        summary_df["VsRefBetterIn"] = summary_df["Agent"].map(better)
        print(f"Paired comparison reference: {ref_name} "
              f"(VsRefMeanDiff = mean score difference per seed, VsRefBetterIn = games won)\n")
    summary_df["SurvivalRate"] = (summary_df["SurvivalRate"] * 100).round(2)

    ranked_df = summary_df.sort_values(
        by=["SurvivalRate", "MeanScore", "MeanTetrises"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    best_agent = ranked_df.iloc[0]
    print("Best Agent (by Survival Rate, then Score, then Tetrises):")
    print(best_agent, "\n")

    print("Agent Rankings (by Survival Rate, Score, Tetrises):")
    show_cols = ["Agent", "SurvivalRate", "MeanScore", "MeanTetrises"] + [
        c for c in ("VsRefMeanDiff", "VsRefBetterIn") if c in ranked_df.columns
    ]
    print(ranked_df[show_cols])

    # ---- Custom plot with manual x positions for maximum tightness ----
    fig, axes = plt.subplots(1, 3, figsize=(9, 4))
    metrics = ["SurvivalRate", "MeanScore", "MeanTetrises"]
    agents_list = list(summary_df["Agent"])
    N = len(agents_list)

    # The "compression factor" (space between bars); <1 squishes bars together
    compress = 0.4   # try 0.2~0.6 for most compact
    bar_width = 0.13 # thin bar

    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        x = np.arange(N) * compress
        ax.bar(x, summary_df[metric], width=bar_width, color='gray', align='center', edgecolor='black')
        ax.set_xticks(x)
        ax.set_xticklabels(agents_list, rotation=20)
        ax.set_title(metric)
        ax.set_xlabel("Agent")
        ax.set_ylabel(metric)
        ax.set_xlim(x[0] - 0.4, x[-1] + 0.4)
        ax.grid(axis='y', linestyle='', linewidth=0.5, alpha=0.5)

    plt.suptitle(f"{run_name} Agent Comparison (Survival > Score > Tetrises)")
    plt.tight_layout(rect=[0, 0.03, 1, 0.93], w_pad=0.5)
    plt.show()

    return ranked_df

# Example usage:
# analyze_agents("from_run_4")

In [ ]:
# Newest gauntlet result folder, e.g. results/gauntlet/from_run_4
analyze_agents(latest_run(GAUNTLET_DIR).name)